# 06 — SARIMA na prática

No notebook anterior, treinamos um ARIMA simples e comparamos com uma baseline.

Agora vamos testar um modelo SARIMA.

A ideia do SARIMA é adicionar uma camada sazonal ao ARIMA, permitindo que o modelo tente capturar ciclos que se repetem ao longo do tempo.

Neste notebook, vamos:

- carregar a série diária;
- repetir a separação treino/teste;
- treinar um modelo SARIMA;
- gerar previsões;
- calcular métricas;
- comparar baseline, ARIMA e SARIMA.

### 06.01 Imports e caminhos

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

import plotly.graph_objects as go

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import (
    CORES,
    aplicar_layout_padrao,
    grafico_linha_padrao
)

caminho_serie_diaria = raiz_projeto / "outputs" / "tabelas" / "serie_diaria_bike.csv"
caminho_metricas_arima = raiz_projeto / "outputs" / "tabelas" / "metricas_arima.csv"
caminho_outputs_tabelas = raiz_projeto / "outputs" / "tabelas"

print("Série diária:")
print(caminho_serie_diaria)

print("\nMétricas anteriores:")
print(caminho_metricas_arima)

### 06.02 Carregar série diária

In [ ]:
serie_diaria = pd.read_csv(caminho_serie_diaria)

serie_diaria["data_hora"] = pd.to_datetime(serie_diaria["data_hora"])

serie_diaria = serie_diaria.sort_values("data_hora").reset_index(drop=True)

print(f"Linhas: {serie_diaria.shape[0]}")
print(f"Colunas: {serie_diaria.shape[1]}")

serie_diaria.head()

## Retomando a série

Vamos usar a mesma série diária trabalhada nos notebooks anteriores.

Assim, conseguimos comparar os modelos usando a mesma base e o mesmo período de teste.

### 06.03 Visualizar série

In [ ]:
fig = grafico_linha_padrao(
    df=serie_diaria,
    x="data_hora",
    y="demanda",
    titulo="Demanda diária de bicicletas",
    labels={
        "data_hora": "Data",
        "demanda": "Demanda diária"
    },
    altura=500
)

fig.show()

### 06.04 Separar treino e teste

In [ ]:
serie_modelagem = serie_diaria[["data_hora", "demanda"]].copy()

tamanho_teste = 60

treino = serie_modelagem.iloc[:-tamanho_teste].copy()
teste = serie_modelagem.iloc[-tamanho_teste:].copy()

print("Tamanho do treino:", treino.shape[0])
print("Tamanho do teste:", teste.shape[0])

print("\nPeríodo de treino:")
print(treino["data_hora"].min(), "até", treino["data_hora"].max())

print("\nPeríodo de teste:")
print(teste["data_hora"].min(), "até", teste["data_hora"].max())

## Estrutura do SARIMA

O SARIMA tem duas camadas:

- uma camada não sazonal: `(p, d, q)`;
- uma camada sazonal: `(P, D, Q, s)`.

O parâmetro `s` representa o tamanho do ciclo sazonal.

Como estamos trabalhando com uma série diária, vamos começar testando um ciclo de 7 períodos, representando um padrão semanal.

### 06.05 Definir parâmetros do SARIMA

In [ ]:
order = (1, 1, 1)
seasonal_order = (1, 0, 1, 7)

print("Parâmetros não sazonais:", order)
print("Parâmetros sazonais:", seasonal_order)

## Treinando o modelo SARIMA

Agora vamos ajustar o modelo usando apenas o período de treino.

Depois, vamos gerar previsões para o mesmo tamanho do período de teste.

### 06.06 Treinar SARIMA

In [ ]:
treino_serie = treino["demanda"].astype(float).reset_index(drop=True)

modelo_sarima = SARIMAX(
    treino_serie,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

resultado_sarima = modelo_sarima.fit(disp=False)

resultado_sarima.summary()

### 06.07 Gerar previsões

In [ ]:
previsao_sarima = resultado_sarima.forecast(steps=len(teste))

teste["previsao_sarima"] = previsao_sarima.values

teste[["data_hora", "demanda", "previsao_sarima"]].head()

### 06.08 Visualizar real versus SARIMA

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=teste["data_hora"],
        y=teste["demanda"],
        mode="lines",
        name="Real",
        line=dict(color=CORES["azul_escuro"], width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=teste["data_hora"],
        y=teste["previsao_sarima"],
        mode="lines",
        name="SARIMA",
        line=dict(color=CORES["azul_principal"], width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Real versus previsto — SARIMA",
    altura=500
)

fig.show()

## Métricas de erro

Vamos usar as mesmas métricas calculadas anteriormente:

- MAE;
- RMSE;
- MAPE.

Assim conseguimos comparar o SARIMA com a baseline e com o ARIMA.

### 06.09 Função de métricas

In [ ]:
def calcular_metricas(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))

    y_real = np.array(y_real)
    y_pred = np.array(y_pred)

    mascara = y_real != 0
    mape = np.mean(
        np.abs((y_real[mascara] - y_pred[mascara]) / y_real[mascara])
    ) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

### 06.10 Calcular métricas do SARIMA

In [ ]:
metricas_sarima = calcular_metricas(
    teste["demanda"],
    teste["previsao_sarima"]
)

df_metricas_sarima = pd.DataFrame([
    {
        "modelo": "SARIMA(1,1,1)(1,0,1,7)",
        **metricas_sarima
    }
])

df_metricas_sarima

### 06.11 Carregar métricas anteriores

In [ ]:
df_metricas_arima = pd.read_csv(caminho_metricas_arima)

df_metricas_arima

### 06.12 Comparar baseline, ARIMA e SARIMA

In [ ]:
df_comparacao_modelos = pd.concat(
    [
        df_metricas_arima,
        df_metricas_sarima
    ],
    ignore_index=True
)

df_comparacao_modelos = df_comparacao_modelos.sort_values("MAPE").reset_index(drop=True)

df_comparacao_modelos

### 06.13 Visualizar comparação de MAPE

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=df_comparacao_modelos["modelo"],
        y=df_comparacao_modelos["MAPE"],
        marker_color=CORES["azul_principal"],
        name="MAPE"
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Comparação dos modelos pelo MAPE",
    altura=450,
    legenda_horizontal=False
)

fig.update_xaxes(title="Modelo")
fig.update_yaxes(title="MAPE (%)")

fig.show()

### 06.14 Resíduos do SARIMA

In [ ]:
teste["residuo_sarima"] = teste["demanda"] - teste["previsao_sarima"]

fig = grafico_linha_padrao(
    df=teste,
    x="data_hora",
    y="residuo_sarima",
    titulo="Resíduos do modelo SARIMA no período de teste",
    labels={
        "data_hora": "Data",
        "residuo_sarima": "Erro"
    },
    altura=450
)

fig.show()

### 06.15 Resultado parcial

In [ ]:
melhor_modelo = df_comparacao_modelos.iloc[0]

print("Melhor modelo nesta comparação:")
print(melhor_modelo["modelo"])

print("\nMAPE:")
print(f"{melhor_modelo['MAPE']:.2f}%")

### 06.16 Salvar resultados

In [ ]:
caminho_previsoes_sarima = caminho_outputs_tabelas / "previsoes_sarima.csv"
caminho_metricas_sarima = caminho_outputs_tabelas / "metricas_sarima.csv"
caminho_comparacao_modelos = caminho_outputs_tabelas / "comparacao_modelos.csv"

teste.to_csv(
    caminho_previsoes_sarima,
    index=False,
    encoding="utf-8-sig"
)

df_metricas_sarima.to_csv(
    caminho_metricas_sarima,
    index=False,
    encoding="utf-8-sig"
)

df_comparacao_modelos.to_csv(
    caminho_comparacao_modelos,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos salvos:")
print("-", caminho_previsoes_sarima)
print("-", caminho_metricas_sarima)
print("-", caminho_comparacao_modelos)

## Próximo passo

Treinamos o SARIMA e comparamos seu desempenho com a baseline e com o ARIMA.

No próximo notebook, vamos consolidar os resultados, escolher o modelo final e gerar uma previsão futura para comunicar o resultado do projeto.